# Chapter 1 — Foundations: SageMath Companion

This is the symbolic, exact-arithmetic counterpart to `code/python/01_foundations.ipynb`. Where the Python notebook uses NumPy floats and pictures, this notebook uses SageMath's exact rational/symbolic arithmetic, native `vector` and `Matrix` constructors, and built-in linear-algebra methods.

**To run this notebook:** open it in [CoCalc](https://cocalc.com/) (no install needed) — *File → Open from URL* and paste the raw GitHub URL — or run locally with the SageMath kernel (`sage -n jupyter`).

**Kernel:** SageMath.

**Layout:**

1. Vectors as exact objects
2. Symbolic linearity check
3. The matrix of a linear map — symbolically
4. Plotting vectors with `arrow2d`
5. Symbolic preview: image, kernel, rank–nullity
6. Affine vs linear — symbolic disproof
7. Vectors over different base rings (QQ, RR, SR, Q[√2])
8. *T(0) = 0* — proved symbolically
9. Composition of linear maps as matrix multiplication
10. **Exercises** for you to solve
11. Solutions

## 1. Vectors as exact objects

Sage's `vector(...)` constructor lets you choose the base ring. Over `QQ` (the rationals) you get exact arithmetic — no floating-point round-off.

In [ ]:
u = vector(QQ, [3, 1])
v = vector(QQ, [1, 2])

show(u, '+', v, '=', u + v)
show(2*u - v)
print('u lives in', u.parent())

## 2. Symbolic linearity check

Where the Python notebook used random sampling to *empirically* test linearity, Sage lets us check it **symbolically** — for *all* inputs at once.

In [ ]:
var('x1 x2 y1 y2 c1 c2')

def is_linear_2to1(f):
    """Symbolically test whether f: R^2 -> R is linear."""
    lhs = f(c1*x1 + c2*y1, c1*x2 + c2*y2)
    rhs = c1*f(x1, x2) + c2*f(y1, y2)
    return bool((lhs - rhs).expand() == 0)

tests = {
    'f(x,y) = 3x + 4y':       lambda a, b: 3*a + 4*b,
    'f(x,y) = 3x + 4y + 7':   lambda a, b: 3*a + 4*b + 7,
    'f(x,y) = x*y':           lambda a, b: a*b,
    'f(x,y) = x^2 + y':       lambda a, b: a^2 + b,
}

for name, f in tests.items():
    print(f'{name:30s}  linear? {is_linear_2to1(f)}')

Note the difference from the Python notebook: there we ran random probes and got *probably linear*. Here we get **provably linear** for all real inputs, because Sage's symbolic engine actually expands and simplifies the expressions.

## 3. The matrix of a linear map — symbolically

Worked Example 1.6 had `T(x, y, z) = (x + 2y, y - z)`. Sage will build the matrix from T's action on the standard basis and verify they agree on a generic input vector.

In [ ]:
def T(v):
    x, y, z = v
    return vector([x + 2*y, y - z])

e1 = vector([1, 0, 0])
e2 = vector([0, 1, 0])
e3 = vector([0, 0, 1])

# Build A by stacking T(e_i) as columns
A = matrix(QQ, [T(e1), T(e2), T(e3)]).transpose()
show(A)

# Symbolic verification on a generic input
var('a b c')
v_sym = vector([a, b, c])
lhs = T(v_sym)
rhs = A * v_sym
show('T(v) =', lhs)
show('A v =', rhs)
print('Equal as symbolic vectors?', bool(lhs == rhs))

## 4. Plotting vectors with Sage's `arrow2d`

The parallelogram rule, redrawn with Sage's plotting (compare to the matplotlib version in the Python notebook).

In [ ]:
u = vector([3, 1])
v = vector([1, 2])
w = u + v

p = (
    arrow2d((0,0), tuple(u), color='blue',  width=2, arrowsize=4)
  + arrow2d((0,0), tuple(v), color='orange', width=2, arrowsize=4)
  + arrow2d(tuple(u), tuple(w), color='orange', width=1, linestyle='dashed')
  + arrow2d(tuple(v), tuple(w), color='blue',   width=1, linestyle='dashed')
  + arrow2d((0,0), tuple(w), color='green', width=2, arrowsize=4)
  + text('u',     tuple(u/2 + vector([0.1, 0.2])),  color='blue',   fontsize=14)
  + text('v',     tuple(v/2 + vector([-0.3, 0.1])), color='orange', fontsize=14)
  + text('u + v', tuple(w + vector([0.1, 0.1])),    color='green',  fontsize=14)
)
p.show(aspect_ratio=1, gridlines=True, frame=True, xmin=-0.5, xmax=5, ymin=-0.5, ymax=4)

## 5. A symbolic preview of what's coming

Sage's real strength shows up in the chapters ahead. As a teaser, here's the same matrix `A` with its image, kernel, and right null space — concepts you'll meet formally in Chapter 5.

In [ ]:
A = matrix(QQ, [[1, 2, 0], [0, 1, -1]])
show('A =', A)
show('image (column space) =', A.column_space())
show('kernel (right null space) =', A.right_kernel())
show('rank =', A.rank())
show('nullity =', A.right_nullity())
print('rank + nullity =', A.rank() + A.right_nullity(), '= number of columns')

Three lines computed the image, kernel, rank, and the rank–nullity theorem, exactly. That's the Sage advantage: linear-algebra primitives are first-class. Future chapters will lean on this heavily.

## 6. Affine vs linear — a symbolic disproof

From the notes (and Worked Example 1.2): `f(x) = mx + b` is *not* linear when `b ≠ 0`. Let's prove it. We compute the residual `f(x+y) − f(x) − f(y)` symbolically — it should be zero for a linear function and nonzero otherwise.

In [ ]:
var('x y m b')

f = lambda t: m*t + b
residual = (f(x + y) - f(x) - f(y)).expand()
show('f(x+y) − f(x) − f(y) =', residual)

# The residual depends only on b. Show it explicitly:
print('Residual when b = 0:', residual.subs(b=0))
print('Residual when b ≠ 0: ', residual, '— this is exactly the constant offset −b.')

Sage just *proved* (not sampled) that the failure of additivity is exactly `−b`. That's the cleanest statement of why affine ≠ linear.

## 7. Vectors over different base rings

The same vector operations work over any field (and most commutative rings). Sage lets you choose. This is one of the key advantages over NumPy, which is always floating-point.

In [ ]:
# Over the rationals — exact
u_QQ = vector(QQ, [1/3, 2/5, 7])
show('Over QQ:', u_QQ, '   3·u =', 3*u_QQ)

# Over the reals — floating point
u_RR = vector(RR, [1/3, 2/5, 7])
show('Over RR:', u_RR, '   3·u =', 3*u_RR)

# Over the symbolic ring — keeps things unevaluated
var('alpha')
u_SR = vector(SR, [alpha, alpha^2, 1])
show('Over SR:', u_SR, '   3·u =', 3*u_SR)

# Over Q[√2] — exact arithmetic with an irrational number
K.<sqrt2> = QuadraticField(2)
u_K = vector(K, [1, sqrt2, 1 + sqrt2])
show('Over Q[√2]:', u_K, '   sqrt2 · u =', sqrt2 * u_K)

Notice in the Q[√2] line: Sage knows `sqrt2 · sqrt2 = 2` *exactly*, not as `2.0000000000000004`.

## 8. Why a linear map must send 0 to 0

From the notes: `f(0) = f(0 + 0) = f(0) + f(0) ⟹ f(0) = 0` for any linear `f`. Let's see Sage confirm this for a generic linear map.

In [ ]:
# Take an arbitrary 2x3 linear map T(v) = M v with symbolic entries
var('m11 m12 m13 m21 m22 m23')
M = matrix(SR, [[m11, m12, m13], [m21, m22, m23]])

zero3 = vector(SR, [0, 0, 0])
show('T(0) =', M * zero3)

print('T(0) is the zero vector regardless of the entries of M.')

## 9. Composition of linear maps as matrix multiplication

Foreshadowing Chapter 4: if `T : ℝ³ → ℝ²` and `S : ℝ² → ℝ²` are linear, then `S ∘ T : ℝ³ → ℝ²` is linear too, and **its matrix is `M_S · M_T`** (in that order). Sage makes the proof a one-liner.

In [ ]:
def T(v):
    x, y, z = v
    return vector([x + 2*y, y - z])

def S(w):
    a, b = w
    return vector([3*a - b, 2*b])

# Build matrices from action on standard basis
M_T = matrix(QQ, [T(vector([1,0,0])), T(vector([0,1,0])), T(vector([0,0,1]))]).transpose()
M_S = matrix(QQ, [S(vector([1,0])),    S(vector([0,1]))]).transpose()

show('M_T =', M_T)
show('M_S =', M_S)
show('M_S · M_T =', M_S * M_T)

# Verify symbolically that (M_S · M_T) · v == S(T(v))
var('a b c')
v_sym = vector([a, b, c])
print('Composition matches?', bool((M_S * M_T) * v_sym == S(T(v_sym))))

Matrix multiplication *is* function composition. We proved it for one example; in Chapter 4 we'll prove it in general.

---

## 10. Exercises

Try each problem before peeking at the Solutions section below. Each prompt has an empty code cell underneath — fill it in.

### Exercise 1 — Exact vector arithmetic
Construct `u = (1/2, 3, −7/4)` and `v = (2, −1, 0)` over `QQ`. Compute `3u − 2v`. Print the result and confirm its parent is `QQ^3`.

In [ ]:
# Your code here


### Exercise 2 — Symbolic linearity tests
Use the `is_linear_2to1` helper from §2 to test whether each of these functions is linear:
- `f(x, y) = x − y`
- `f(x, y) = (x + y) / 2`
- `f(x, y) = sqrt(2)·x + y/3`
- `f(x, y) = x − y + 1`

In [ ]:
# Your code here


### Exercise 3 — Matrix of a linear map
Define `T(x, y) = (2x + y, x − 3y, 4y)`. Build the matrix `A` of `T` from its action on the standard basis of `ℝ²`. Verify symbolically (using a generic vector `(a, b)`) that `A · v == T(v)`.

In [ ]:
# Your code here


### Exercise 4 — Solving a linear system
Using the matrix `A = [[1, 2, 0], [0, 1, −1]]` from §5, find a vector `v ∈ QQ^3` such that `A · v = (5, 0)`. (Hint: `A.solve_right(...)`.) Then use the kernel of `A` from §5 to describe **all** such vectors.

In [ ]:
# Your code here


### Exercise 5 — High-dimensional sanity check
Build the vector `u ∈ QQ^10` whose i-th component is `i` (so `u = (1, 2, 3, …, 10)`). Compute `sum(u)` and check it equals `55`. Then compute `2u` and confirm `sum(2u) == 110`.

In [ ]:
# Your code here


### Exercise 6 — Affine fails, by exactly how much?
The function `f(x) = 3x + 5` is affine, not linear. Compute the residual `f(x + y) − f(x) − f(y)` symbolically using Sage and report the value. Then compute `f(c·x) − c·f(x)` to see how homogeneity also fails.

In [ ]:
# Your code here


### Exercise 7 — Plot scalar multiples
Given `w = (2, 1)`, draw the vectors `−1.5·w`, `−0.5·w`, `1·w`, `1.5·w`, `2.5·w` from the origin (use Sage's `arrow2d` and color them via `rainbow(5)`). Confirm visually that they all lie on the same line through the origin.

In [ ]:
# Your code here


---

## 11. Solutions

*Try the exercises before reading these.*

### Solution 1

In [ ]:
u = vector(QQ, [1/2, 3, -7/4])
v = vector(QQ, [2, -1, 0])
result = 3*u - 2*v
show(result)
print('parent =', result.parent())

### Solution 2

In [ ]:
tests = {
    'f(x,y) = x - y':              lambda a, b: a - b,
    'f(x,y) = (x + y)/2':          lambda a, b: (a + b)/2,
    'f(x,y) = sqrt(2)*x + y/3':    lambda a, b: sqrt(2)*a + b/3,
    'f(x,y) = x - y + 1':          lambda a, b: a - b + 1,
}
for name, f in tests.items():
    print(f'{name:35s}  linear? {is_linear_2to1(f)}')

### Solution 3

In [ ]:
def T_ex3(v):
    x, y = v
    return vector([2*x + y, x - 3*y, 4*y])

A = matrix(QQ, [T_ex3(vector([1, 0])), T_ex3(vector([0, 1]))]).transpose()
show('A =', A)

var('a b')
v_sym = vector([a, b])
print('A · v =', A * v_sym)
print('T(v) =', T_ex3(v_sym))
print('Equal?', bool(A * v_sym == T_ex3(v_sym)))

### Solution 4

In [ ]:
A = matrix(QQ, [[1, 2, 0], [0, 1, -1]])
b = vector(QQ, [5, 0])
v_particular = A.solve_right(b)
show('A particular solution: v =', v_particular)
show('A · v =', A * v_particular)

# All solutions are v_particular + (anything in the kernel)
print('General solution = (', v_particular, ') + t · (', A.right_kernel().basis()[0], ') for t ∈ QQ')

### Solution 5

In [ ]:
u = vector(QQ, [i for i in range(1, 11)])
show(u)
print('sum(u)  =', sum(u))
print('sum(2u) =', sum(2*u))
assert sum(u) == 55 and sum(2*u) == 110

### Solution 6

In [ ]:
var('x y c')
f = lambda t: 3*t + 5
additivity_residual = (f(x + y) - f(x) - f(y)).expand()
homogeneity_residual = (f(c*x) - c*f(x)).expand()
show('Additivity residual: f(x+y) − f(x) − f(y) =', additivity_residual)
show('Homogeneity residual: f(cx) − c·f(x) =', homogeneity_residual)
print('Both nonzero, so f is not linear.')

### Solution 7

In [ ]:
w = vector([2, 1])
scalars = [-1.5, -0.5, 1, 1.5, 2.5]
colors = rainbow(len(scalars))

p = sum(arrow2d((0, 0), tuple(s * w), color=c, width=2)
        for s, c in zip(scalars, colors))
p += text('All scalar multiples of w lie on one line through the origin', (0, -2.5), fontsize=11)
p.show(aspect_ratio=1, gridlines=True, frame=True, xmin=-5, xmax=6, ymin=-3, ymax=4)

---

## Where to next

- **Chapter 2 (Sage):** vector geometry — magnitude, dot product, projections — using exact symbolic angles.
- **Chapter 3 (Sage):** `A.rref()` and `A.solve_right(b)` for systems.
- **Chapter 9 (Sage):** `A.eigenvalues()`, `A.eigenvectors_right()`, `A.jordan_form()`.